# Deploy torchcast-decision-12b on an Amazon SageMaker AI real-time endpoint and invoke it

> **Note:** This is sample code for demonstration purposes only and is not intended for production use without
> additional security testing and review.

[torchcast-decision-12b](https://huggingface.co/torchcast-ai/torchcast-decision-12b) is TorchCast AI's decision
model, Gemma 4 12B with a LoRA fine-tune merged into the weights. It answers typed questions about a piece of text
(the "state") with probabilities. A `choice` question returns a distribution over named options, a `noul` question the
probability of yes, and a `score` question an expected level on an ordered scale. Each question is one forward pass
over the prompt that reads the probabilities of the option letters at the answer position; the model generates no
text.

> **Licence of the weights:** the model weights are licensed **CC BY-NC 4.0, for non-commercial use only**. This
> notebook downloads them onto your endpoint. Read the
> [model's licence](https://huggingface.co/torchcast-ai/torchcast-decision-12b/blob/main/LICENSE) before you run it,
> and do not serve commercial traffic with it unless TorchCast AI grants you other terms.

The serving image is the AWS vLLM Deep Learning Container (SageMaker flavour) plus TorchCast's own decision server
(MIT), which `container/serve.py` puts behind SageMaker's `/ping` and `/invocations` routes:

```
InvokeEndpoint (SigV4) ──► SageMaker ──► :8080 /invocations   container/serve.py + TorchCast decision_server.py
                                              │  one vLLM request per question
                                              ▼
                                        127.0.0.1:8890  vLLM 0.30.0, torchcast-decision-12b BF16, 16,384-token context (GPU)
```

**Prerequisites** (details in the README): Docker with `buildx`, `curl` and the AWS CLI v2 on the machine running
this notebook, about 60 GB of free disk for the image build, the IAM permissions in `notebook-user-policy.json`, and a
SageMaker quota of at least 1 for `ml.g6e.xlarge for endpoint usage`.
**Run time:** about 1 hour, plus the first image build, which pulls a base image of several GB.
**Cost:** `ml.g6e.xlarge` is $2.6054 per hour in us-east-1, us-east-2 and us-west-2, plus ECR storage for the image.
The clean-up cell at the end deletes the endpoint and everything else this notebook created.

## 1. Install the Python packages

In [ ]:
%pip install -q --disable-pip-version-check -r requirements.txt

## 2. Configuration
Everything this notebook creates carries the tag `created-by=torchcast-decision-12b-sagemaker-sample`, and the
clean-up deletes only resources with that tag, so a resource with the same name that someone else made is left alone.

Set `WEIGHTS_S3_URI` to an S3 prefix holding the model files (see the README) to load the weights from S3 with
network isolation on. Left at `None`, the endpoint downloads them from Hugging Face at the pinned revision, which is
what we benchmarked.

In [ ]:
import json
import os
import shutil
import subprocess  # nosec B404
import time
from datetime import datetime, timedelta, timezone

import boto3
import pandas as pd
from botocore.config import Config
from botocore.exceptions import ClientError

import decisions

REGION = "us-east-2"
INSTANCE_TYPE = "ml.g6e.xlarge"                # 1x NVIDIA L40S 48 GB; the 24 GB of BF16 weights fit on one GPU
NAME = "torchcast-decision-12b"                # the model, endpoint config and endpoint
REPO = "torchcast-decision-12b-sagemaker"      # private ECR repository for the serving image
ROLE_NAME = "torchcast-decision-12b-sagemaker-execution"
EXISTING_IMAGE = None                          # an image URI build.sh already pushed, to skip the build
EXISTING_ROLE_ARN = None                       # a SageMaker execution role to use instead of creating one
WEIGHTS_S3_URI = None                          # e.g. "s3://<bucket>/torchcast-decision-12b/49107b55/"
DELETE_REPOSITORY = True                       # clean-up also deletes the ECR repository, if this notebook made it
PROFILE = None                                 # a named AWS profile, or None for the default credential chain
TAG = {"Key": "created-by", "Value": "torchcast-decision-12b-sagemaker-sample"}

session = boto3.Session(region_name=REGION, profile_name=PROFILE)
account = session.client("sts").get_caller_identity()["Account"]
sm, ecr, iam = session.client("sagemaker"), session.client("ecr"), session.client("iam")
# One client for every invocation in this notebook. Its connection pool must hold at least as many connections
# as the highest concurrency below (botocore's default is 10), and retries are off so that every failed request
# is counted instead of being hidden by a retry.
smr = session.client("sagemaker-runtime", config=Config(
    max_pool_connections=128, connect_timeout=10, read_timeout=70, tcp_keepalive=True,
    retries={"total_max_attempts": 1, "mode": "standard"}))


def show(x):
    """The value as text with the account ID masked, so saved outputs do not carry it."""
    return str(x).replace(account, "<account-id>")


def describe(call, **kw):
    """The resource's description, or None when it does not exist."""
    try:
        return call(**kw)
    except ClientError as e:
        if e.response["Error"]["Code"] == "ValidationException" and "Could not find" in e.response["Error"]["Message"]:
            return None
        raise


def ours(arn):
    """True when the SageMaker resource carries this notebook's tag."""
    return TAG in sm.list_tags(ResourceArn=arn)["Tags"]


print(f"{INSTANCE_TYPE} endpoint {NAME} in {REGION}, account {show(account)}")

## 3. Build and push the serving image
`container/build.sh` downloads TorchCast's serving files from the model's GitHub repository at the v1.0.0 commit,
checks them against pinned SHA-256 sums, and builds `container/Dockerfile` on the AWS vLLM DLC. The image tag is a
hash of everything the build uses, so an unchanged image is never rebuilt and a changed one never overwrites a tag in
use.

SageMaker accepts only a Docker image manifest v2 schema 2. Recent Docker versions push OCI media types and add
provenance and SBOM attestations, which turn the push into an OCI image index that SageMaker rejects, so the
script builds with `docker buildx --provenance=false --sbom=false --output type=image,oci-mediatypes=false,push=true`.
This cell checks the manifest media type in ECR again after the push.

The cell first creates the repository (immutable tags, scan on push, tagged), or stops if a repository with that
name exists without this notebook's tag.

In [ ]:
try:
    ecr.create_repository(repositoryName=REPO, imageTagMutability="IMMUTABLE",
                          imageScanningConfiguration={"scanOnPush": True},
                          encryptionConfiguration={"encryptionType": "AES256"}, tags=[TAG])
    print("created ECR repository", REPO)
except ecr.exceptions.RepositoryAlreadyExistsException:
    repo = ecr.describe_repositories(repositoryNames=[REPO])["repositories"][0]
    if TAG not in ecr.list_tags_for_resource(resourceArn=repo["repositoryArn"])["tags"]:
        raise SystemExit(f"an ECR repository named {REPO} exists and was not created by this notebook: "
                         "change REPO") from None

IMAGE = EXISTING_IMAGE
if IMAGE is None:
    env = {**os.environ, "REGION": REGION, "ACCOUNT": account, "REPO": REPO, "AWS_DEFAULT_REGION": REGION}
    if PROFILE:
        env["AWS_PROFILE"] = PROFILE
    t0 = time.time()
    # A fixed argument list and no shell: the only inputs are the settings above, passed as environment variables.
    proc = subprocess.Popen([shutil.which("bash"), "container/build.sh"], env=env, text=True,  # nosec B603  # noqa: S603
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in proc.stdout:
        if line.startswith("IMAGE="):
            IMAGE = line.strip().split("=", 1)[1]
        print(show(line.rstrip())[:200])
    if proc.wait() != 0 or not IMAGE:
        raise SystemExit("build.sh failed: see the lines above")
    print(f"build.sh finished in {(time.time() - t0) / 60:.0f} min")

image_tag = IMAGE.rsplit(":", 1)[1]
media = ecr.batch_get_image(repositoryName=REPO, imageIds=[{"imageTag": image_tag}])["images"][0][
    "imageManifestMediaType"]
if media != "application/vnd.docker.distribution.manifest.v2+json":
    raise SystemExit(f"SageMaker cannot use an image with manifest media type {media}")
print("image", show(IMAGE), "| manifest", media)
try:
    scan = ecr.describe_image_scan_findings(repositoryName=REPO, imageId={"imageTag": image_tag})
    print("ECR scan:", scan["imageScanStatus"]["status"],
          scan.get("imageScanFindings", {}).get("findingSeverityCounts"))
except ecr.exceptions.ScanNotFoundException:
    print("ECR scan: not started yet")

## 4. Execution role (least privilege)
The endpoint pulls this one image and writes its logs and metrics. With `WEIGHTS_S3_URI` set it may also read that
one prefix. The trust policy names this account in `aws:SourceAccount`, so SageMaker resources in other accounts
cannot use the role. Skipped if you set `EXISTING_ROLE_ARN`.

In [ ]:
ROLE_ARN = EXISTING_ROLE_ARN
if ROLE_ARN is None:
    trust = {"Version": "2012-10-17", "Statement": [{"Effect": "Allow", "Action": "sts:AssumeRole",
             "Principal": {"Service": "sagemaker.amazonaws.com"},
             "Condition": {"StringEquals": {"aws:SourceAccount": account}}}]}
    statements = [
        {"Effect": "Allow", "Action": "ecr:GetAuthorizationToken", "Resource": "*"},
        {"Effect": "Allow", "Action": ["ecr:BatchGetImage", "ecr:GetDownloadUrlForLayer",
                                       "ecr:BatchCheckLayerAvailability"],
         "Resource": f"arn:aws:ecr:{REGION}:{account}:repository/{REPO}"},
        {"Effect": "Allow", "Action": ["logs:CreateLogGroup", "logs:CreateLogStream", "logs:PutLogEvents",
                                       "logs:DescribeLogStreams"],
         "Resource": f"arn:aws:logs:{REGION}:{account}:log-group:/aws/sagemaker/*"},
        {"Effect": "Allow", "Action": "cloudwatch:PutMetricData", "Resource": "*",
         "Condition": {"StringLike": {"cloudwatch:namespace": ["/aws/sagemaker/*", "AWS/SageMaker*"]}}}]
    if WEIGHTS_S3_URI:
        bucket, prefix = WEIGHTS_S3_URI.removeprefix("s3://").split("/", 1)
        statements += [
            {"Effect": "Allow", "Action": "s3:GetObject", "Resource": f"arn:aws:s3:::{bucket}/{prefix}*",
             "Condition": {"StringEquals": {"s3:ResourceAccount": account}}},
            {"Effect": "Allow", "Action": "s3:ListBucket", "Resource": f"arn:aws:s3:::{bucket}",
             "Condition": {"StringLike": {"s3:prefix": [f"{prefix}*"]},
                           "StringEquals": {"s3:ResourceAccount": account}}}]
    policy = {"Version": "2012-10-17", "Statement": statements}
    try:
        ROLE_ARN = iam.create_role(RoleName=ROLE_NAME, AssumeRolePolicyDocument=json.dumps(trust), Tags=[TAG],
                                   Description="torchcast-decision-12b SageMaker endpoint")["Role"]["Arn"]
    except iam.exceptions.EntityAlreadyExistsException:
        if TAG not in iam.list_role_tags(RoleName=ROLE_NAME)["Tags"]:
            raise SystemExit(f"a role named {ROLE_NAME} exists and was not created by this notebook") from None
        ROLE_ARN = iam.get_role(RoleName=ROLE_NAME)["Role"]["Arn"]
        iam.update_assume_role_policy(RoleName=ROLE_NAME, PolicyDocument=json.dumps(trust))
    iam.put_role_policy(RoleName=ROLE_NAME, PolicyName="endpoint", PolicyDocument=json.dumps(policy))
    time.sleep(15)   # IAM changes take a few seconds to reach SageMaker
print("execution role", show(ROLE_ARN))

## 5. Create the model, endpoint config and endpoint

| Setting | Value | Why |
|---|---|---|
| `TP`, `MAX_MODEL_LEN`, `GPU_UTIL`, `PREFIX_CACHE` | `1`, `16384`, `0.90`, `on` | TorchCast's published vLLM settings; the model fits one 48 GB GPU |
| `MAX_PARALLEL` | `256` | vLLM requests the decision server keeps in flight; its default of 8 would cap throughput |
| `NODELAY` | `on` | TCP_NODELAY on the server's connections; without it each response waits 40-60 ms for a delayed ACK |
| `RoutingStrategy` | `LEAST_OUTSTANDING_REQUESTS` | sends each request to the least busy instance once you scale out |
| start-up timeouts | 1800 s | the image pull, the 24 GB weight download and vLLM's compile take longer than the defaults |
| network isolation | on only with `WEIGHTS_S3_URI` | without it the container downloads the weights from Hugging Face |

The cell reuses an endpoint this notebook already created that is `InService` or still `Creating`, so you can run it
again after a kernel restart. In our run the container started 16 minutes after `CreateEndpoint`; the endpoint turns
`InService` after that, once vLLM has downloaded and loaded the weights and `/ping` answers 200.

In [ ]:
ENV = {"TP": "1", "MAX_MODEL_LEN": "16384", "GPU_UTIL": "0.90", "MAX_PARALLEL": "256",
       "PREFIX_CACHE": "on", "NODELAY": "on"}

endpoint = describe(sm.describe_endpoint, EndpointName=NAME)
if endpoint and not ours(endpoint["EndpointArn"]):
    raise SystemExit(f"an endpoint named {NAME} exists and was not created by this notebook: change NAME")
if endpoint and endpoint["EndpointStatus"] not in ("Creating", "InService"):
    raise SystemExit(f"{NAME} is {endpoint['EndpointStatus']}: run the clean-up cell, then this cell again")
if endpoint is None:
    if describe(sm.describe_endpoint_config, EndpointConfigName=NAME) or describe(sm.describe_model, ModelName=NAME):
        raise SystemExit(f"a model or endpoint config named {NAME} is left over: run the clean-up cell first")
    container = {"Image": IMAGE, "Environment": ENV}
    if WEIGHTS_S3_URI:
        container["ModelDataSource"] = {"S3DataSource": {"S3Uri": WEIGHTS_S3_URI.rstrip("/") + "/",
                                                         "S3DataType": "S3Prefix", "CompressionType": "None"}}
    sm.create_model(ModelName=NAME, ExecutionRoleArn=ROLE_ARN, PrimaryContainer=container,
                    EnableNetworkIsolation=bool(WEIGHTS_S3_URI), Tags=[TAG])
    sm.create_endpoint_config(EndpointConfigName=NAME, Tags=[TAG], ProductionVariants=[{
        "VariantName": "AllTraffic", "ModelName": NAME, "InstanceType": INSTANCE_TYPE, "InitialInstanceCount": 1,
        "ContainerStartupHealthCheckTimeoutInSeconds": 1800, "ModelDataDownloadTimeoutInSeconds": 1800,
        "RoutingConfig": {"RoutingStrategy": "LEAST_OUTSTANDING_REQUESTS"}}])
    sm.create_endpoint(EndpointName=NAME, EndpointConfigName=NAME, Tags=[TAG])
    print("CreateEndpoint", NAME)

t0, last = time.time(), None
while True:
    endpoint = sm.describe_endpoint(EndpointName=NAME)
    status = endpoint["EndpointStatus"]
    if status != last:
        print(f"{(time.time() - t0) / 60:5.1f} min  {status}")
        last = status
    if status in ("InService", "Failed"):
        break
    if time.time() - t0 > 75 * 60:
        raise SystemExit("still not InService after 75 minutes: check the endpoint's CloudWatch logs")
    time.sleep(30)
if status != "InService":
    raise SystemExit(show(f"{NAME} failed: {endpoint.get('FailureReason', '')}"))

## 6. Invoke it
The body is TorchCast's `/v1/systemone` request: a `state` and named `questions`. Any SigV4-signed client works;
here it is boto3's `sagemaker-runtime` client. The response has one answer per question and the token usage.

`decisions.check` verifies the shape of every answer: the asked type, probabilities over exactly the asked options
that sum to 1, the `choice` equal to the most probable option, and the `score` inside its scale.

In [ ]:
def invoke(body: dict) -> dict:
    r = smr.invoke_endpoint(EndpointName=NAME, ContentType="application/json", Accept="application/json",
                            Body=json.dumps(body))
    return json.loads(r["Body"].read())


body = decisions.triage_request("I was charged twice for order #4411 and the app still shows it as unpaid. "
                                "I need the second charge refunded before my card limit runs out on Friday.")
t0 = time.perf_counter()
answer = invoke(body)
print(f"{(time.perf_counter() - t0) * 1000:.0f} ms from this machine, including the network path to {REGION}")
print(json.dumps(answer, indent=1))
problems = decisions.check(body, answer)
if problems:
    raise RuntimeError(f"malformed response: {problems}")
print("team:", answer["answers"]["team"]["choice"], "| urgent:", round(answer["answers"]["urgent"]["noul"], 3),
      "| severity:", round(answer["answers"]["severity"]["score"], 2))

# A request the decision server cannot answer (here an unknown question type) is the caller's error. It comes back
# as HTTP 422 from the container, which boto3 raises as ModelError.
try:
    invoke({"state": "hi", "questions": {"q": {"type": "maybe"}}})
    raise RuntimeError("a malformed request was accepted")
except smr.exceptions.ModelError as e:
    print("malformed request ->", e.response.get("OriginalStatusCode"),
          show(e.response.get("OriginalMessage", ""))[:200])

## 7. Concurrency: latency percentiles at 1, 10 and 64 requests in flight
Each level keeps a fixed number of requests in flight (a closed loop: a thread sends its next request when the
previous one returns) and records each request's latency from this machine. Settings that affect the numbers:

* Distinct prompts. vLLM caches the prefix of every prompt it has seen. Repeating one request would mostly measure
  that cache, so `decisions.tickets()` writes a new support ticket for every request, and the cell checks
  that no state repeats across the warm-up and all three levels. Only the system prompt and chat template are shared,
  as they would be in production.
* One client with enough connections. The `sagemaker-runtime` client from section 2 has a pool of 128 connections.
  With botocore's default of 10, a level of 64 would queue requests inside the client and report that wait as
  endpoint latency.
* No retries. Retries are off, so every failed request appears in `errors` instead of as a slow success.
* A warm-up of 64 requests at 64 in flight that is not measured, so the client has opened its connections before
  the first level starts.

Each request asks one question (the team), so requests per second equal decisions per second and compare with the
measured table in the README. A request with *k* questions costs *k* forward passes. Latency includes the network
path from this machine to the Region: run the notebook in the same Region (for example on an EC2 instance) to
measure the endpoint rather than your connection.

In [ ]:
LEVELS = {1: 200, 10: 400, 64: 1280}          # concurrency: requests sent at that level
gen = decisions.tickets()
seen = set()


def draw(n):
    batch = [next(gen) for _ in range(n)]
    seen.update(state for state, _labels in batch)
    return batch


warm = draw(64)
decisions.run_level(invoke, [decisions.routing_request(s) for s, _ in warm], 64)

window_start = datetime.now(timezone.utc)
levels, labelled = [], {}
for c, n in LEVELS.items():
    batch = draw(n)
    bodies = [decisions.routing_request(s) for s, _ in batch]
    result = decisions.run_level(invoke, bodies, c)
    levels.append(result)
    labelled[c] = (bodies, [labels for _s, labels in batch], result["responses"])
    print(f"c={c:3}: {result['ok']}/{n} ok in {result['wall_s']:.1f} s, p50 {result['p50_ms']:.1f} ms")
window_end = datetime.now(timezone.utc)
total = 64 + sum(LEVELS.values())
if len(seen) != total:
    raise RuntimeError(f"{total - len(seen)} prompts repeated; the prefix cache would flatter the results")
print(f"{total} requests, all with distinct states")

table = pd.DataFrame([{k: r[k] for k in ("concurrency", "requests", "errors", "decisions_per_s",
                                          "p10_ms", "p50_ms", "p90_ms", "p99_ms")} for r in levels])
table.set_index("concurrency").round(1)

Every response is checked, and because each synthetic ticket was written for one team, the cell also reports how
often the model routes it there. That agreement is a sanity check of the deployment on easy synthetic text, not a
measure of the model's accuracy.

In [ ]:
checks = []
for c, (bodies, labels, responses) in labelled.items():
    answered = [(b, lab, r) for b, lab, r in zip(bodies, labels, responses, strict=True) if r is not None]
    malformed = [p for b, _lab, r in answered for p in decisions.check(b, r)]
    well_formed = [(lab, r) for b, lab, r in answered if not decisions.check(b, r)]
    agree = sum(r["answers"]["team"]["choice"] == lab["team"] for lab, r in well_formed)
    checks.append({"concurrency": c, "answered": len(answered), "malformed": len(malformed),
                   "team agrees with label": f"{agree / max(1, len(answered)):.1%}"})
    if malformed:
        print(f"c={c}: first malformed response: {malformed[0]}")
for r in levels:
    for sample in r["error_samples"]:
        print(f"c={r['concurrency']} error: {show(sample)}")
if any(row["malformed"] for row in checks) or any(r["errors"] for r in levels):
    raise RuntimeError("some requests failed or returned malformed answers: see the lines above")
pd.DataFrame(checks).set_index("concurrency")

## 8. SageMaker's view of the same requests
`ModelLatency` is the time inside the container and `OverheadLatency` SageMaker's own share, both as
request-weighted averages over the three levels, in milliseconds. The rest of the client-side latency above is the
network path and the client itself. Metrics arrive a minute or two late, so the cell waits first.

In [ ]:
time.sleep(150)
cw = session.client("cloudwatch")
dims = [{"Name": "EndpointName", "Value": NAME}, {"Name": "VariantName", "Value": "AllTraffic"}]
metrics = {}
for name in ("ModelLatency", "OverheadLatency", "Invocations", "Invocation4XXErrors", "Invocation5XXErrors"):
    stat = "Average" if name.endswith("Latency") else "Sum"
    points = cw.get_metric_statistics(Namespace="AWS/SageMaker", MetricName=name, Dimensions=dims, Period=60,
                                      StartTime=window_start.replace(second=0, microsecond=0),
                                      EndTime=window_end.replace(second=0, microsecond=0) + timedelta(minutes=1),
                                      Statistics=[stat, "SampleCount"])["Datapoints"]
    if not points:
        metrics[name] = None
    elif stat == "Average":    # microseconds, weighted by requests per minute, to milliseconds
        weight = sum(p["SampleCount"] for p in points)
        metrics[name] = sum(p["Average"] * p["SampleCount"] for p in points) / weight / 1000
    else:
        metrics[name] = sum(p["Sum"] for p in points)
pd.Series(metrics, name="value").round(2)

## 9. Clean up
Deletes the endpoint (this stops the hourly charge), its endpoint config and model, its CloudWatch log group, the
execution role and, with `DELETE_REPOSITORY = True`, the ECR repository with the image. Each is deleted only if it
carries this notebook's tag. The cell needs only the configuration cell (section 2), so it also works after a kernel
restart, and it skips anything already gone.

In [ ]:
endpoint = describe(sm.describe_endpoint, EndpointName=NAME)
if endpoint and ours(endpoint["EndpointArn"]):
    sm.delete_endpoint(EndpointName=NAME)
    sm.get_waiter("endpoint_deleted").wait(EndpointName=NAME, WaiterConfig={"Delay": 15, "MaxAttempts": 80})
    print("deleted endpoint", NAME)
    logs = session.client("logs")
    try:
        logs.delete_log_group(logGroupName=f"/aws/sagemaker/Endpoints/{NAME}")
        print("deleted log group", f"/aws/sagemaker/Endpoints/{NAME}")
    except logs.exceptions.ResourceNotFoundException:
        pass
elif endpoint:
    print(f"kept endpoint {NAME}: not created by this notebook")

config = describe(sm.describe_endpoint_config, EndpointConfigName=NAME)
if config and ours(config["EndpointConfigArn"]):
    sm.delete_endpoint_config(EndpointConfigName=NAME)
    print("deleted endpoint config", NAME)
model = describe(sm.describe_model, ModelName=NAME)
if model and ours(model["ModelArn"]):
    sm.delete_model(ModelName=NAME)
    print("deleted model", NAME)

if EXISTING_ROLE_ARN is None:
    try:
        if TAG in iam.list_role_tags(RoleName=ROLE_NAME)["Tags"]:
            for policy_name in iam.list_role_policies(RoleName=ROLE_NAME)["PolicyNames"]:
                iam.delete_role_policy(RoleName=ROLE_NAME, PolicyName=policy_name)
            iam.delete_role(RoleName=ROLE_NAME)
            print("deleted role", ROLE_NAME)
    except iam.exceptions.NoSuchEntityException:
        pass

if DELETE_REPOSITORY:
    try:
        repo = ecr.describe_repositories(repositoryNames=[REPO])["repositories"][0]
        if TAG in ecr.list_tags_for_resource(resourceArn=repo["repositoryArn"])["tags"]:
            ecr.delete_repository(repositoryName=REPO, force=True)
            print("deleted ECR repository", REPO)
    except ecr.exceptions.RepositoryNotFoundException:
        pass